# 第 21 天：多因子合成

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：多因子合成
> 必做：等权合成
> 选做：IC加权
> 目标产出：多因子模型

第 21-30 天是整套 30 天计划的“项目段”。  
前 20 天你已经学了因子定义、标签、IC、分层、价值/质量/动量/波动率/流动性、预处理和 Alpha101 复现。  
现在开始，重点从“会做一个因子”变成“会管理一批因子，并能交付一份像样的研究项目”。



## 0. 今天你要真正学会什么？

1. 理解为什么不能只追求单个最强因子。
2. 掌握等权合成、IC 加权、滚动 IC 加权三种基础方法。
3. 学会检查合成因子的 IC、分层收益、换手和因子贡献。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。

## 1. 先建立直觉

单因子像一个研究员的观点，多因子模型像一场投委会。等权合成是每个观点都有一票，IC 加权是让历史上更靠谱的观点多说两句。

一个成熟的因子研究员不会只问“这个因子收益高不高”。  
他会继续问：

- 它在什么市场状态下有效？
- 它和已有因子是否重复？
- 它的换手是否能承受？
- 它的收益来自 Alpha，还是来自行业、风格、容量和数据偏差？
- 它能否被写进一个别人看得懂、能复现、能维护的研究报告？

## 2. 今日研究框架


明确研究问题
  -> 准备数据和标签
  -> 构建候选因子或模型
  -> 统一预处理
  -> IC / 分层 / 换手检验
  -> 风险和稳定性检查
  -> 写入研究结论
  -> 决定保留、观察或淘汰


## 3. 准备统一研究环境

下面的模拟数据会贯穿第 21-30 天。  
真实研究时，你可以把这里替换为自己的行情、财务、行业和股票池数据。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(202630)

dates = pd.bdate_range("2021-01-04", periods=520)
assets = [f"S{i:03d}" for i in range(1, 81)]

industries = pd.Series(
    np.random.choice(["消费", "科技", "制造", "医药", "金融", "周期"], size=len(assets)),
    index=assets,
    name="industry",
)

size_base = pd.Series(np.random.normal(0, 1, len(assets)), index=assets, name="size")
value_base = pd.Series(np.random.normal(0, 1, len(assets)), index=assets, name="value")
quality_base = pd.Series(np.random.normal(0, 1, len(assets)), index=assets, name="quality")
growth_base = pd.Series(np.random.normal(0, 1, len(assets)), index=assets, name="growth")
beta_base = pd.Series(np.random.uniform(0.75, 1.35, len(assets)), index=assets, name="beta")

market_ret = np.random.normal(0.00025, 0.010, len(dates))
value_premium = np.random.normal(0.00005, 0.0030, len(dates))
quality_premium = np.random.normal(0.00004, 0.0025, len(dates))
growth_premium = np.random.normal(0.00002, 0.0028, len(dates))
industry_shock = pd.DataFrame(
    np.random.normal(0, 0.004, (len(dates), industries.nunique())),
    index=dates,
    columns=sorted(industries.unique()),
)

daily_ret = pd.DataFrame(index=dates, columns=assets, dtype=float)
for asset in assets:
    ind = industries[asset]
    daily_ret[asset] = (
        beta_base[asset] * market_ret
        + value_base[asset] * value_premium * 0.18
        + quality_base[asset] * quality_premium * 0.16
        + growth_base[asset] * growth_premium * 0.12
        + industry_shock[ind].to_numpy()
        + np.random.normal(0, 0.017, len(dates))
    )

close = 25 * np.exp(daily_ret.cumsum())
overnight = pd.DataFrame(np.random.normal(0, 0.005, close.shape), index=dates, columns=assets)
open_ = close.shift(1) * (1 + overnight)
open_.iloc[0] = close.iloc[0] * (1 + overnight.iloc[0])
span = pd.DataFrame(np.random.uniform(0.004, 0.035, close.shape), index=dates, columns=assets)
high = pd.DataFrame(np.maximum(open_, close) * (1 + span), index=dates, columns=assets)
low = pd.DataFrame(np.minimum(open_, close) * (1 - span), index=dates, columns=assets)
vwap = (open_ + high + low + close) / 4

base_volume = 1_000_000 * np.exp(size_base)
volume = pd.DataFrame(index=dates, columns=assets, dtype=float)
for asset in assets:
    volume[asset] = (
        base_volume[asset]
        * np.random.lognormal(0, 0.45, len(dates))
        * (1 + daily_ret[asset].abs().to_numpy() * 18)
    )

returns = close.pct_change()
future_5d = close.shift(-5) / close - 1
future_20d = close.shift(-20) / close - 1

def noisy_static(series: pd.Series, scale: float = 0.25) -> pd.DataFrame:
    noise = np.random.normal(0, scale, (len(dates), len(assets)))
    return pd.DataFrame(series.values[None, :] + noise, index=dates, columns=assets)

raw_factors = {
    "value": noisy_static(value_base, 0.28),
    "quality": noisy_static(quality_base, 0.24),
    "growth": noisy_static(growth_base, 0.30),
    "size": noisy_static(size_base, 0.18),
    "momentum_20": close.pct_change(20),
    "momentum_60": close.pct_change(60),
    "low_vol": -returns.rolling(20).std(),
    "liquidity": np.log(volume.rolling(20).mean()),
    "reversal_5": -close.pct_change(5),
    "price_volume": -returns.rolling(10).corr(volume.pct_change()),
}

print("模拟研究环境：")
print({
    "dates": len(dates),
    "assets": len(assets),
    "industries": industries.nunique(),
    "factor_count": len(raw_factors),
})
print("\n行业分布：")
print(industries.value_counts().sort_index())


## 4. 准备统一研究工具箱

这套函数会帮助你把因子研究固定成流程：预处理、IC、分层、多空、合成、面板数据、简单模型和组合权重。


In [ ]:
def winsorize_cs(df: pd.DataFrame, lower: float = 0.01, upper: float = 0.99) -> pd.DataFrame:
    low_q = df.quantile(lower, axis=1)
    high_q = df.quantile(upper, axis=1)
    return df.T.clip(lower=low_q, upper=high_q, axis=1).T


def zscore_cs(df: pd.DataFrame) -> pd.DataFrame:
    mean = df.mean(axis=1)
    std = df.std(axis=1).replace(0, np.nan)
    return df.sub(mean, axis=0).div(std, axis=0)


def preprocess_factor(df: pd.DataFrame) -> pd.DataFrame:
    return zscore_cs(winsorize_cs(df.replace([np.inf, -np.inf], np.nan)))


def preprocess_library(library: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    return {name: preprocess_factor(factor) for name, factor in library.items()}


def rank_ic(factor: pd.DataFrame, label: pd.DataFrame) -> pd.Series:
    label = label.reindex_like(factor)
    return factor.rank(axis=1).corrwith(label.rank(axis=1), axis=1)


def long_short_return(factor: pd.DataFrame, label: pd.DataFrame, q: float = 0.2) -> pd.Series:
    pct_rank = factor.rank(axis=1, pct=True)
    long_leg = label.where(pct_rank >= 1 - q).mean(axis=1)
    short_leg = label.where(pct_rank <= q).mean(axis=1)
    return (long_leg - short_leg).dropna()


def top_bucket_turnover(factor: pd.DataFrame, q: float = 0.2) -> pd.Series:
    top = factor.rank(axis=1, pct=True).ge(1 - q).astype(float)
    return top.diff().abs().mean(axis=1).dropna()


def factor_report(factor: pd.DataFrame, label: pd.DataFrame, name: str) -> pd.Series:
    clean_factor = factor.replace([np.inf, -np.inf], np.nan)
    ic = rank_ic(clean_factor, label).dropna()
    ls = long_short_return(clean_factor, label)
    turnover = top_bucket_turnover(clean_factor)
    return pd.Series({
        "factor": name,
        "ic_mean": ic.mean(),
        "ic_ir": ic.mean() / ic.std() if ic.std() else np.nan,
        "ic_positive_ratio": (ic > 0).mean(),
        "long_short_mean": ls.mean(),
        "long_short_vol": ls.std(),
        "long_short_sharpe_like": ls.mean() / ls.std() * np.sqrt(52) if ls.std() else np.nan,
        "turnover": turnover.mean(),
        "valid_days": len(ic),
    })


def summarize_library(library: dict[str, pd.DataFrame], label: pd.DataFrame) -> pd.DataFrame:
    rows = [factor_report(factor, label, name) for name, factor in library.items()]
    return pd.DataFrame(rows).set_index("factor").sort_values("ic_mean", ascending=False)


def neutralize_by_industry(factor: pd.DataFrame, groups: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=factor.index, columns=factor.columns, dtype=float)
    for _, cols in groups.groupby(groups).groups.items():
        cols = list(cols)
        block = factor[cols]
        out[cols] = block.sub(block.mean(axis=1), axis=0)
    return out


def make_equal_weight_composite(library: dict[str, pd.DataFrame], names: list[str]) -> pd.DataFrame:
    aligned = [library[name] for name in names]
    return pd.concat(aligned, keys=names).groupby(level=1).mean()


def make_weighted_composite(library: dict[str, pd.DataFrame], weights: pd.Series) -> pd.DataFrame:
    total = None
    for name, weight in weights.items():
        piece = library[name] * weight
        total = piece if total is None else total.add(piece, fill_value=0)
    return total


def rolling_ic_weights(library: dict[str, pd.DataFrame], label: pd.DataFrame, window: int = 60) -> pd.DataFrame:
    ic_table = pd.DataFrame({name: rank_ic(factor, label) for name, factor in library.items()})
    rolling_mean = ic_table.rolling(window).mean()
    abs_sum = rolling_mean.abs().sum(axis=1).replace(0, np.nan)
    return rolling_mean.div(abs_sum, axis=0).fillna(0)


def build_panel(library: dict[str, pd.DataFrame], label: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for name, factor in library.items():
        frames.append(factor.stack().rename(name))
    panel = pd.concat(frames, axis=1)
    panel["label"] = label.stack()
    panel.index.names = ["date", "asset"]
    return panel.replace([np.inf, -np.inf], np.nan).dropna()


def time_split_panel(panel: pd.DataFrame, split_ratio: float = 0.7):
    unique_dates = panel.index.get_level_values("date").unique().sort_values()
    split_date = unique_dates[int(len(unique_dates) * split_ratio)]
    train = panel.loc[panel.index.get_level_values("date") < split_date]
    test = panel.loc[panel.index.get_level_values("date") >= split_date]
    return train, test, split_date


def fit_ridge(X: np.ndarray, y: np.ndarray, lam: float = 5.0) -> np.ndarray:
    X_ = np.column_stack([np.ones(len(X)), X])
    penalty = np.eye(X_.shape[1]) * lam
    penalty[0, 0] = 0
    return np.linalg.solve(X_.T @ X_ + penalty, X_.T @ y)


def predict_ridge(X: np.ndarray, coef: np.ndarray) -> np.ndarray:
    X_ = np.column_stack([np.ones(len(X)), X])
    return X_ @ coef


def make_market_neutral_weights(score: pd.DataFrame, q: float = 0.2) -> pd.DataFrame:
    pct = score.rank(axis=1, pct=True)
    long_mask = pct >= 1 - q
    short_mask = pct <= q
    long_w = long_mask.div(long_mask.sum(axis=1).replace(0, np.nan), axis=0)
    short_w = short_mask.div(short_mask.sum(axis=1).replace(0, np.nan), axis=0)
    return (long_w - short_w).fillna(0)


def portfolio_return(weights: pd.DataFrame, next_return: pd.DataFrame) -> pd.Series:
    return (weights.shift(1) * next_return).sum(axis=1).dropna()


factor_library = preprocess_library(raw_factors)
base_summary = summarize_library(factor_library, future_5d)
print("基础因子库摘要：")
print(base_summary[["ic_mean", "ic_ir", "ic_positive_ratio", "turnover"]].round(4))


## 5. 今日核心实验


### 实验 1：等权合成：先把每个声音调到同一音量

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
selected = ["value", "quality", "momentum_20", "low_vol", "liquidity"]
equal_composite = make_equal_weight_composite(factor_library, selected)

equal_report = factor_report(equal_composite, future_5d, "equal_composite")
print(equal_report.drop("factor").astype(float).round(4))


### 实验 2：静态 IC 加权：让历史更稳定的因子权重大一点

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
summary = summarize_library({name: factor_library[name] for name in selected}, future_5d)
raw_weight = summary["ic_mean"].copy()
ic_weight = raw_weight / raw_weight.abs().sum()
ic_composite = make_weighted_composite(factor_library, ic_weight)

print("IC 权重：")
print(ic_weight.round(4))
print("\nIC 加权组合：")
print(factor_report(ic_composite, future_5d, "ic_weighted").drop("factor").astype(float).round(4))


### 实验 3：滚动 IC 加权：只用过去窗口，不偷看未来

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
rolling_weights = rolling_ic_weights({name: factor_library[name] for name in selected}, future_5d, window=80)
rolling_composite = pd.DataFrame(0.0, index=dates, columns=assets)

for name in selected:
    rolling_composite = rolling_composite.add(factor_library[name].mul(rolling_weights[name], axis=0), fill_value=0)

compare = pd.DataFrame([
    factor_report(equal_composite, future_5d, "equal"),
    factor_report(ic_composite, future_5d, "static_ic_weighted"),
    factor_report(rolling_composite, future_5d, "rolling_ic_weighted"),
]).set_index("factor")

print(compare[["ic_mean", "ic_ir", "long_short_mean", "turnover"]].round(4))


### 实验 4：合成因子分层收益曲线

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
curve = pd.DataFrame({
    "equal": long_short_return(equal_composite, future_5d).cumsum(),
    "static_ic": long_short_return(ic_composite, future_5d).cumsum(),
    "rolling_ic": long_short_return(rolling_composite, future_5d).cumsum(),
})

curve.plot(figsize=(10, 4), title="多因子合成多空累计收益")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()
plt.close()


### 实验 5：贡献拆解：多因子模型不是黑箱

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
contrib = {}
for name in selected:
    contrib[name] = rank_ic(factor_library[name], future_5d).mean()
contrib = pd.Series(contrib).sort_values(ascending=False)

print("单因子 IC 贡献观察：")
print(contrib.round(4))

weight_table = pd.DataFrame({
    "single_ic": contrib,
    "static_weight": ic_weight.reindex(contrib.index),
    "latest_rolling_weight": rolling_weights.iloc[-1].reindex(contrib.index),
})
print("\n权重与贡献表：")
print(weight_table.round(4))


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：多因子合成
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：直接把原始因子相加，忘记方向和量纲。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：用全样本 IC 做权重，造成隐蔽的未来函数。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：只保留历史最好因子，导致组合过拟合。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：忽略相关性，多个相似因子叠在一起其实还是一个声音。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 22 天会研究因子衰减，判断信号到底能活多久。

## 13. 一句话收尾

多因子合成 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本、严格样本外检验和风险约束。
